# SupplyChainIQ Statistical Analysis

This notebook tests business hypotheses on the cleaned supply chain datasets and quantifies relationships between key variables.

**Note:** The executable version of this analysis is `run_statistical_analysis.py`. Verified results are in `statistical_analysis_results.json`.

In [ ]:
import math
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style='whitegrid')
ROOT = Path('..') if Path('../data/processed').exists() else Path('.')
base = ROOT / 'data' / 'processed'
charts = ROOT / 'notebooks' / 'charts_stats'
charts.mkdir(parents=True, exist_ok=True)

d1 = pd.read_csv(base / 'supplychainiq_dataset1_clean.csv', low_memory=False)
d2 = pd.read_csv(base / 'supplychainiq_dataset2_clean.csv')

for col in ['order_date', 'shipping_date']:
    if col in d1.columns:
        d1[col] = pd.to_datetime(d1[col], errors='coerce')

print(f'Dataset 1: {d1.shape[0]:,} rows x {d1.shape[1]} cols')
print(f'Dataset 2: {d2.shape[0]:,} rows x {d2.shape[1]} cols')
d1[['sales', 'benefit_per_order', 'profit_margin_pct', 'late_delivery_flag']].head()

In [ ]:
# Helper: Cramer's V
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct)[0]
    n = ct.to_numpy().sum()
    r, k = ct.shape
    return math.sqrt(chi2 / (n * (min(r, k) - 1)))

# Hypothesis 1: Late shipments do not change profit margin.
on_time = d1[d1['late_delivery_flag'] == 0]['profit_margin_pct'].dropna()
late = d1[d1['late_delivery_flag'] == 1]['profit_margin_pct'].dropna()
welch_t = stats.ttest_ind(on_time, late, equal_var=False, nan_policy='omit')
print(f'H1 Welch t-test: t={welch_t.statistic:.4f}, p={welch_t.pvalue:.6f}')
print(f'  On-time: n={len(on_time):,}, mean={on_time.mean():.4f}%')
print(f'  Late:    n={len(late):,}, mean={late.mean():.4f}%')

# Hypothesis 2: Shipping mode and late-delivery risk are independent.
shipping_ct = pd.crosstab(d1['shipping_mode'], d1['late_delivery_flag'])
chi2, chi_p, chi_dof, chi_expected = stats.chi2_contingency(shipping_ct)
cv = cramers_v(shipping_ct)
print(f'\nH2 Chi-square: chi2={chi2:.4f}, p={chi_p:.2e}, dof={chi_dof}')
print(f'   Cramer\'s V: {cv:.4f}')
print(shipping_ct)

# Hypothesis 3: Defect rates do not differ by inspection result.
pass_def = d2[d2['inspection_result'] == 'Pass']['defect_rate_raw'].dropna()
fail_def = d2[d2['inspection_result'] == 'Fail']['defect_rate_raw'].dropna()
pending_def = d2[d2['inspection_result'] == 'Pending']['defect_rate_raw'].dropna()
kw = stats.kruskal(pass_def, fail_def, pending_def)
print(f'\nH3 Kruskal-Wallis: H={kw.statistic:.4f}, p={kw.pvalue:.6f}')
for name, grp in [('Pass', pass_def), ('Fail', fail_def), ('Pending', pending_def)]:
    print(f'  {name}: n={len(grp)}, mean={grp.mean():.4f}, median={grp.median():.4f}')

In [ ]:
# Correlations: Pearson for Dataset 1, Spearman for Dataset 2.
d1_corr = d1[['sales', 'benefit_per_order', 'order_item_discount_rate',
              'order_item_quantity', 'delivery_delay_days', 'profit_margin_pct']].corr(method='pearson')
d2_corr = d2[['stock_levels', 'products_sold', 'revenue_generated',
              'shipping_cost', 'lead_time_gap_days']].corr(method='spearman')

print('Dataset 1 Pearson Correlation Matrix:')
print(d1_corr.round(4).to_string())
print('\nDataset 2 Spearman Correlation Matrix:')
print(d2_corr.round(4).to_string())

In [ ]:
# Chart 1: Dataset 1 Pearson correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(d1_corr, annot=True, cmap='Blues', ax=ax, fmt='.2f', linewidths=0.5)
ax.set_title('Dataset 1: Pearson Correlation Heatmap')
fig.tight_layout(); fig.savefig(charts / 'correlation_heatmap_dataset1.png', dpi=160); plt.show()

# Chart 2: Dataset 2 Spearman correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(d2_corr, annot=True, cmap='Greens', ax=ax, fmt='.2f', linewidths=0.5)
ax.set_title('Dataset 2: Spearman Correlation Heatmap')
fig.tight_layout(); fig.savefig(charts / 'correlation_heatmap_dataset2.png', dpi=160); plt.show()

# Chart 3: Profit margin by late delivery flag
d1_plot = d1.copy()
d1_plot['late_delivery_flag'] = d1_plot['late_delivery_flag'].astype(str)
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=d1_plot, x='late_delivery_flag', y='profit_margin_pct',
            hue='late_delivery_flag', legend=False, ax=ax,
            palette={'0': '#22c55e', '1': '#ef4444'})
ax.set_title('Profit Margin by Late Delivery Flag (Welch t-test)')
ax.set_xlabel('Late Delivery Flag (0=On-Time, 1=Late)')
ax.set_ylabel('Profit Margin %')
fig.tight_layout(); fig.savefig(charts / 'profit_margin_by_late_flag.png', dpi=160); plt.show()

# Chart 4: Shipping mode vs late delivery (stacked bar)
ct = pd.crosstab(d1['shipping_mode'], d1['late_delivery_flag'], normalize='index')
fig, ax = plt.subplots(figsize=(9, 6))
ct.plot(kind='bar', stacked=True, ax=ax, color=['#22c55e', '#ef4444'])
ax.set_title('Shipping Mode vs Late Delivery Risk (Chi-square)')
ax.set_xlabel('Shipping Mode'); ax.set_ylabel('Proportion')
ax.legend(['On-Time', 'Late'], title='Delivery')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
fig.tight_layout(); fig.savefig(charts / 'shipping_mode_vs_late_delivery.png', dpi=160); plt.show()

# Chart 5: Defect rate by inspection result
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=d2, x='inspection_result', y='defect_rate_raw',
            hue='inspection_result', legend=False,
            order=['Pass', 'Fail', 'Pending'], hue_order=['Pass', 'Fail', 'Pending'],
            ax=ax, palette={'Pass': '#22c55e', 'Fail': '#ef4444', 'Pending': '#f59e0b'})
ax.set_title('Defect Rate by Inspection Result (Kruskal-Wallis)')
ax.set_xlabel('Inspection Result'); ax.set_ylabel('Defect Rate (raw)')
fig.tight_layout(); fig.savefig(charts / 'defect_rate_by_inspection.png', dpi=160); plt.show()

# Chart 6: Stock levels vs products sold scatter
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=d2, x='stock_levels', y='products_sold', hue='product_type', ax=ax, s=80, alpha=0.8)
ax.set_title('Stock Levels vs Products Sold (Spearman)')
fig.tight_layout(); fig.savefig(charts / 'stock_vs_sold_scatter.png', dpi=160); plt.show()

print('Charts saved:', sorted(p.name for p in charts.iterdir()))

## Summary

| Hypothesis | Test | Result | p-value |
|---|---|---|---|
| Late shipments reduce profit margin | Welch t-test | Not significant | 0.268 |
| Shipping mode affects late-delivery risk | Chi-square + Cramer's V | **Significant** (V=0.457) | <0.001 |
| Defect rates differ by inspection result | Kruskal-Wallis | Not significant | 0.345 |

Detailed results and statistics are stored in `statistical_analysis_results.json`.